# Разбор матча: интерактивная витрина визуализаций

Тестовый ноутбук поверх слоя **gold** платформы: выбираете лигу, сезон, матч, команду и игрока —
графики во вкладках перерисовываются. Сами графики интерактивные (plotly): наведение показывает
игрока, минуту, xG и исход действия, колёсико и рамка — масштаб, двойной клик — сброс.
Только `pandas` + `plotly` + `ipywidgets`, без футбольных библиотек. Данные читаются из Trino:

| Что | Таблица gold | Источник |
|---|---|---|
| События с координатами (SPADL, поле 0–100 × 0–100, атака слева направо) | `fct_event` | WhoScored |
| Удары с xG | `fct_shot` | Understat |
| Командная статистика по матчам | `fct_team_match` | FotMob / FBref |
| Статистика игроков по матчам | `fct_player_match` | FotMob / FBref |
| Справочники | `dim_match`, `dim_team`, `dim_player` | — |

Вкладки:
1. **Losses** — потери в своей трети и какие из них привели к удару соперника
2. **xG timeline** — кумулятивный xG матча и территориальный «моментум»
3. **Pass map** — карта передач игрока и его перцентили по лиге за сезон
4. **Shots & entries** — удары и входы в 20-метровую зону у ворот
5. **Pass network** — сети передач обеих команд
6. **League form** — изменение формы всех команд лиги: последние 7 туров против начала сезона

Запускайте все ячейки (Run → Run All Cells); панель выбора и вкладки — в конце.
Графики живут в виджетах, поэтому на GitHub они не видны — снимки есть на лендинге.

In [1]:
import os
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd
import trino
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as w
from IPython.display import display

warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype"  # рендер расширением JupyterLab, без встраивания plotly.js в файл

# В JupyterHub платформы переменные TRINO_* уже в окружении (docs/ANALYST_ONBOARDING.md).
# TRINO_VERIFY=false — только для локального запуска с самоподписанным сертификатом.
_conn = trino.dbapi.connect(
    host=os.environ.get("TRINO_HOST", "trino"),
    port=int(os.environ.get("TRINO_PORT", "443")),
    user=os.environ.get("TRINO_USER", "analyst_svc"),
    http_scheme="https",
    auth=trino.auth.BasicAuthentication(
        os.environ.get("TRINO_USER", "analyst_svc"), os.environ["TRINO_PASSWORD"]
    ),
    verify=os.environ.get("TRINO_VERIFY", "true").lower() != "false",
    catalog="iceberg",
    schema="gold",
)


def q(sql: str) -> pd.DataFrame:
    cur = _conn.cursor()
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])


def sql_list(values) -> str:
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)

## Стиль и поле

Тёмная тема; поле рисуется фигурами plotly в единицах WhoScored (0–100) с пропорциями 105 × 68 м.

In [2]:
BG, PANEL, LINE, TEXT, MUTED = "#0f1218", "#161b24", "#6b7386", "#e8eaf0", "#8d95a8"
OK, BAD, GOLD = "#2dd4bf", "#f87171", "#fbbf24"
HOME_C, AWAY_C = "#f5f5f5", "#7dd3fc"  # хозяева / гости

PITCH_L, PITCH_W = 105.0, 68.0  # метры
KX, KY = 100 / PITCH_L, 100 / PITCH_W

BASE_LAYOUT = dict(
    paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(color=TEXT, family="Helvetica Neue, Helvetica, Arial, sans-serif", size=12),
    hoverlabel=dict(bgcolor="#1f2530", font=dict(color=TEXT, size=12), bordercolor=LINE),
    legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=-0.02, x=0),
    margin=dict(l=50, r=20, t=80, b=95),
)


def dist_to_goal_m(x, y):
    xm, ym = np.asarray(x) / KX, np.asarray(y) / KY
    return np.hypot(PITCH_L - xm, PITCH_W / 2 - ym)


def _ellipse_path(cx, cy, rx, ry, t0=0.0, t1=2 * np.pi, n=40):
    t = np.linspace(t0, t1, n)
    pts = [f"{cx + rx * np.cos(a):.2f},{cy + ry * np.sin(a):.2f}" for a in t]
    return "M " + " L ".join(pts)


def pitch_shapes(xref="x", yref="y", lc=LINE):
    sh = []
    ln = dict(color=lc, width=1)

    def rect(x0, y0, x1, y1):
        sh.append(dict(type="rect", x0=x0, y0=y0, x1=x1, y1=y1, line=ln, xref=xref, yref=yref))

    def path(p):
        sh.append(dict(type="path", path=p, line=ln, xref=xref, yref=yref))

    rect(0, 0, 100, 100)
    sh.append(dict(type="line", x0=50, y0=0, x1=50, y1=100, line=ln, xref=xref, yref=yref))
    for x0, sgn in ((0, 1), (100, -1)):
        rect(x0, 50 - 20.16 * KY, x0 + sgn * 16.5 * KX, 50 + 20.16 * KY)
        rect(x0, 50 - 9.16 * KY, x0 + sgn * 5.5 * KX, 50 + 9.16 * KY)
        rect(x0, 50 - 3.66 * KY, x0 - sgn * 2, 50 + 3.66 * KY)  # ворота
        px = x0 + sgn * 11 * KX
        a0, a1 = (-0.925, 0.925) if sgn > 0 else (np.pi - 0.925, np.pi + 0.925)
        path(_ellipse_path(px, 50, 9.15 * KX, 9.15 * KY, a0, a1, 20))
        sh.append(dict(type="circle", x0=px - 0.4, y0=49.4, x1=px + 0.4, y1=50.6, line=ln,
                       fillcolor=lc, xref=xref, yref=yref))
    path(_ellipse_path(50, 50, 9.15 * KX, 9.15 * KY) + " Z")
    sh.append(dict(type="circle", x0=49.6, y0=49.4, x1=50.4, y1=50.6, line=ln, fillcolor=lc,
                   xref=xref, yref=yref))
    return sh


def axis_names(fig, row, col):
    ref = fig._grid_ref[row - 1][col - 1][0].trace_kwargs
    return ref["xaxis"], ref["yaxis"]


def pitch_axes(fig, row, col, xlim=(-3, 103)):
    xa, _ = axis_names(fig, row, col)
    fig.update_xaxes(range=list(xlim), showgrid=False, zeroline=False, showticklabels=False,
                     visible=False, row=row, col=col)
    fig.update_yaxes(range=[-3, 103], showgrid=False, zeroline=False, showticklabels=False,
                     visible=False, scaleanchor=xa, scaleratio=PITCH_W / PITCH_L, row=row, col=col)


def pitch_subplots(titles, xlim=(-3, 103), height=520):
    fig = make_subplots(rows=1, cols=len(titles), subplot_titles=[t or " " for t in titles],
                        horizontal_spacing=0.04)
    for i in range(len(titles)):
        xa, ya = axis_names(fig, 1, i + 1)
        fig.update_layout(shapes=list(fig.layout.shapes) + pitch_shapes(xa, ya))
        pitch_axes(fig, 1, i + 1, xlim)
    fig.update_layout(height=height, **BASE_LAYOUT)
    fig.update_annotations(font=dict(size=13, color=TEXT), xanchor="left", x=0.0)
    for i, ann in enumerate(fig.layout.annotations):  # заголовки панелей — слева над панелью
        ann.x = fig.layout[f"xaxis{'' if i == 0 else i + 1}"].domain[0]
    return fig


def add_arrows(fig, df, color, xa="x", ya="y", width=1.2, opacity=0.85, hover=None, name=None,
               showlegend=False):
    # стрелки — аннотациями (без hover) + невидимые маркеры на концах для подсказки
    anns = [dict(x=r.end_x, y=r.end_y, ax=r.x, ay=r.y, xref=xa, yref=ya, axref=xa, ayref=ya,
                 showarrow=True, arrowhead=2, arrowsize=1.1, arrowwidth=width, arrowcolor=color,
                 opacity=opacity, text="")
            for r in df.itertuples()]
    fig.update_layout(annotations=list(fig.layout.annotations) + anns)
    col = 1 if xa == "x" else int(xa[1:])
    if hover is not None and len(df):
        fig.add_trace(go.Scatter(x=df.end_x, y=df.end_y, mode="markers",
                                 marker=dict(size=7, color=color, opacity=0.001),
                                 hovertext=hover, hoverinfo="text", showlegend=False), row=1, col=col)
    if showlegend:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color=color, width=2),
                                 name=name or "", showlegend=True), row=1, col=col)
    return fig


def suptitle(fig, text):
    fig.update_layout(title=dict(text=f"<b>{text}</b>", x=0.01, xanchor="left", font=dict(size=16)))
    return fig


def note(fig, text):
    fig.add_annotation(text=text, xref="paper", yref="paper", x=0, y=0, yshift=-62, yanchor="top", showarrow=False,
                       font=dict(size=10, color=MUTED), xanchor="left", align="left")
    return fig

## Загрузка данных

Всё кэшируется по ключу (лига/сезон/матч), чтобы смена выбора не ходила в Trino повторно.

In [3]:
coverage = q('''
SELECT league, season, count(DISTINCT match_id) AS matches
FROM fct_event WHERE league NOT LIKE 'INT-%' GROUP BY 1, 2 ORDER BY 1, 2
''')
LEAGUES = sorted(coverage.league.unique())


def seasons_of(league):
    return sorted(coverage[coverage.league == league].season, reverse=True)


@lru_cache(maxsize=32)
def matches_of(league, season):
    df = q(f'''
    SELECT m.match_id, m.match_date, h.team_name AS home_name, a.team_name AS away_name,
           m.home_score, m.away_score
    FROM dim_match m
    JOIN dim_team h ON h.team_id = m.home_team_id
    JOIN dim_team a ON a.team_id = m.away_team_id
    WHERE m.league = '{league}' AND m.season = '{season}' AND m.is_completed
      AND m.match_id IN (SELECT DISTINCT match_id FROM fct_event
                         WHERE league = '{league}' AND season = '{season}')
    ORDER BY m.match_date DESC, h.team_name
    ''')
    df["label"] = [f"{r.match_date:%d.%m.%Y}  {r.home_name} {r.home_score}–{r.away_score} {r.away_name}"
                   for r in df.itertuples()]
    return df


@lru_cache(maxsize=16)
def load_match(match_id):
    meta = q(f'''
    SELECT m.match_id, m.match_date, m.league, m.season, m.gameweek,
           m.home_team_id, h.team_name AS home_name, m.home_score,
           m.away_team_id, a.team_name AS away_name, m.away_score
    FROM dim_match m
    JOIN dim_team h ON h.team_id = m.home_team_id
    JOIN dim_team a ON a.team_id = m.away_team_id
    WHERE m.match_id = '{match_id}'
    ''').iloc[0]
    events = q(f'''
    SELECT event_id, team_id, player_id, period, minute, x, y, end_x, end_y,
           action, _action_source_note AS note, outcome_success AS success
    FROM fct_event
    WHERE match_id = '{match_id}' AND period IN ('FirstHalf', 'SecondHalf')
    ORDER BY event_id
    ''').reset_index(drop=True)
    events["seq"] = np.arange(len(events))
    events["player_id"] = events.player_id.fillna("")
    shots = q(f'''
    SELECT s.team_id, s.player_id, p.player_name, s.minute, s.x * 100 AS x, s.y * 100 AS y,
           s.xg, s.situation, s.result, s.is_goal
    FROM fct_shot s LEFT JOIN dim_player p ON p.player_id = s.player_id
    WHERE s.match_id = '{match_id}'
    ORDER BY s.minute
    ''')
    shots = shots.reindex(columns=["team_id", "player_id", "player_name", "minute", "x", "y",
                                   "xg", "situation", "result", "is_goal"]) \
        .astype({"x": float, "y": float, "xg": float, "minute": float})
    shots["is_goal"] = shots.is_goal.fillna(False).astype(bool)
    pm = q(f'''
    SELECT player_id, team_id, minutes_played, passes FROM fct_player_match
    WHERE match_id = '{match_id}'
    ''').reindex(columns=["player_id", "team_id", "minutes_played", "passes"])
    pm = pm.dropna(subset=["player_id"])
    ids = sorted(set(events.player_id[events.player_id.ne("")]) | set(pm.player_id))
    names = q(f"SELECT player_id, player_name FROM dim_player WHERE player_id IN ({sql_list(ids)})") \
        if ids else pd.DataFrame(columns=["player_id", "player_name"])
    # список игроков: по событиям (есть всегда), минуты — из fct_player_match, если есть
    players = (events[events.player_id.ne("")].groupby(["player_id", "team_id"])
               .agg(actions=("seq", "size"), passes=("action", lambda a: (a == "pass").sum()))
               .reset_index()
               .merge(pm[["player_id", "minutes_played"]], on="player_id", how="left")
               .merge(names, on="player_id", how="left"))
    players["player_name"] = players.player_name.fillna(players.player_id)
    return {
        "meta": meta, "events": events, "shots": shots, "players": players,
        "home": meta.home_team_id, "away": meta.away_team_id,
        "names": {meta.home_team_id: meta.home_name, meta.away_team_id: meta.away_name},
        "colors": {meta.home_team_id: HOME_C, meta.away_team_id: AWAY_C},
        "title": f"{meta.home_name} {meta.home_score}–{meta.away_score} {meta.away_name} · "
                 f"{meta.league.split('-', 1)[1]} · {meta.match_date:%d.%m.%Y}",
    }


@lru_cache(maxsize=8)
def season_team_matches(league, season):
    return q(f'''
    WITH sx AS (
      SELECT match_id, team_id,
             sum(CASE WHEN situation = 'open_play' THEN xg ELSE 0 END) AS op_xg
      FROM fct_shot WHERE league = '{league}' AND season = '{season}' GROUP BY 1, 2
    )
    SELECT t.team_id, d.short_name, t.date, t.gameweek, t.opponent_id,
           coalesce(f.op_xg, 0) AS op_xg_for, coalesce(a.op_xg, 0) AS op_xg_against
    FROM fct_team_match t
    JOIN dim_team d ON d.team_id = t.team_id
    LEFT JOIN sx f ON f.match_id = t.match_id AND f.team_id = t.team_id
    LEFT JOIN sx a ON a.match_id = t.match_id AND a.team_id = t.opponent_id
    WHERE t.league = '{league}' AND t.season = '{season}' AND t.is_completed
    ORDER BY t.team_id, t.date
    ''')


@lru_cache(maxsize=8)
def season_player_metrics(league, season, min_minutes=900):
    sp = q(f'''
    SELECT player_id, sum(minutes_played) AS minutes,
           sum(passes) AS passes, sum(passes_completed) AS passes_completed,
           sum(key_passes) AS key_passes, sum(xa) AS xa, sum(xg_chain) AS xg_chain,
           sum(xg_buildup) AS xg_buildup, sum(ball_recoveries) AS ball_recoveries,
           sum(tackles) + sum(interceptions) AS tackles_int, sum(dribbles_won) AS dribbles_won
    FROM fct_player_match
    WHERE league = '{league}' AND season = '{season}'
    GROUP BY 1 HAVING sum(minutes_played) >= {min_minutes}
    ''')
    if sp.empty:
        return None, None

    def per90(col):
        return sp[col] / sp.minutes * 90

    metrics = pd.DataFrame({
        "Passes /90": per90("passes"),
        "Pass accuracy %": sp.passes_completed / sp.passes * 100,
        "Key passes /90": per90("key_passes"),
        "xA /90": per90("xa"),
        "xG chain /90": per90("xg_chain"),
        "xG buildup /90": per90("xg_buildup"),
        "Ball recoveries /90": per90("ball_recoveries"),
        "Tackles + int. /90": per90("tackles_int"),
        "Dribbles won /90": per90("dribbles_won"),
    })
    metrics.index = sp.player_id
    return metrics, metrics.rank(pct=True) * 100

## Графики

Каждая функция получает загруженный матч `M` и возвращает фигуру plotly. Определения:
потеря — неудачная передача/кросс/ввод, неудачный обыгрыш или `bad_touch` на своих 40 % поля;
«привела к удару» — соперник бил в следующих 10 действиях; вход в зону — передача/кросс,
завершившийся в 20 м от центра ворот; территория — доля действий команды в финальной трети
за 5-минутный отрезок.

In [4]:
LOSS_PASS = {"pass", "cross", "throw_in", "goalkick", "freekick_short", "freekick_crossed",
             "corner_short", "corner_crossed"}
SHOT_ACTIONS = ["shot", "shot_freekick"]


def _names(M):
    return M["players"].set_index("player_id").player_name


def plot_losses(M):
    ev, names = M["events"], _names(M)
    is_loss = (
        ev.action.eq("bad_touch")
        | (ev.action.isin(LOSS_PASS) & ~ev.success.fillna(True))
        | (ev.action.eq("take_on") & ~ev.success.fillna(True))
    )
    losses = ev[is_loss & (ev.x <= 40)].copy()

    def led_to_shot(row, window=10):
        nxt = ev.iloc[row.seq + 1 : row.seq + 1 + window]
        nxt = nxt[nxt.period == row.period]
        return bool(((nxt.team_id != row.team_id) & nxt.action.isin(SHOT_ACTIONS)).any())

    losses["led_to_shot"] = losses.apply(led_to_shot, axis=1) if len(losses) else False
    losses["kind"] = np.where(losses.action.eq("bad_touch"), losses.note.fillna("bad touch"), losses.action)
    fig = pitch_subplots(["", ""])
    for i, team in enumerate((M["home"], M["away"])):
        col = i + 1
        xa, ya = axis_names(fig, 1, col)
        fig.add_shape(type="rect", x0=0, y0=0, x1=40, y1=100, fillcolor=BAD, opacity=0.06,
                      line=dict(width=0), xref=xa, yref=ya, layer="below")
        d = losses[losses.team_id == team]
        hover = [f"{names.get(r.player_id, r.player_id)} · {int(r.minute)}' · {r.kind}"
                 + (" · → opponent shot" if r.led_to_shot else "") for r in d.itertuples()]
        d = d.assign(hover=hover)
        plain, star = d[~d.led_to_shot], d[d.led_to_shot]
        fig.add_trace(go.Scatter(x=plain.x, y=plain.y, mode="markers", name="loss",
                                 marker=dict(size=9, color=M["colors"][team], opacity=0.8, line=dict(color=BG, width=0.5)),
                                 hovertext=plain.hover, hoverinfo="text", showlegend=(i == 0), legendgroup="loss"), row=1, col=col)
        fig.add_trace(go.Scatter(x=star.x, y=star.y, mode="markers", name="loss → opponent shot ≤10 actions",
                                 marker=dict(size=18, symbol="star", color=BAD, line=dict(color=BG, width=0.5)),
                                 hovertext=star.hover, hoverinfo="text", showlegend=(i == 0), legendgroup="star"), row=1, col=col)
        fig.layout.annotations[i].text = f"<b>{M['names'][team]}: {len(d)} losses in own third, {len(star)} led to a shot</b>"
        fig.add_annotation(text="own goal", x=3, y=94, xref=xa, yref=ya, showarrow=False,
                           font=dict(size=10, color=MUTED), xanchor="left")
    suptitle(fig, M["title"])
    note(fig, "Losses: unsuccessful pass/cross/throw-in/take-on, bad touch, dispossessed, error · own 40 % of the pitch · gold.fct_event (WhoScored)")
    return fig


def plot_timeline(M, BIN=5):
    ev, shots = M["events"], M["shots"]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.05)
    max_min = int(max(ev.minute.max(), shots.minute.max() if len(shots) else 0)) + 1
    for team in (M["home"], M["away"]):
        s = shots[shots.team_id == team].sort_values("minute").copy()
        s["cum"] = s.xg.cumsum()
        x = np.concatenate([[0], s.minute.values, [max_min]])
        y = np.concatenate([[0], s.cum.values, [s.xg.sum()]])
        fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(shape="hv", color=M["colors"][team], width=2.5),
                                 name=f"{M['names'][team]}  {s.xg.sum():.2f} xG", hoverinfo="skip"), row=1, col=1)
        hover = [f"{r.player_name} · {int(r.minute)}' · xG {r.xg:.2f} · {r.result} ({r.situation})" for r in s.itertuples()]
        fig.add_trace(go.Scatter(x=s.minute, y=s.cum, mode="markers", showlegend=False,
                                 marker=dict(size=6, color=M["colors"][team]), hovertext=hover, hoverinfo="text"), row=1, col=1)
        g = s[s.is_goal]
        fig.add_trace(go.Scatter(x=g.minute, y=g.cum, mode="markers+text", showlegend=False,
                                 marker=dict(size=13, color=GOLD, line=dict(color=BG, width=1)),
                                 text=[f"{r.player_name} {int(r.minute)}'" for r in g.itertuples()],
                                 textposition=[("top left" if r.minute > max_min - 15 else "top right")
                                               if (k % 2 == 0) != (team == M["away"]) else
                                               ("bottom left" if r.minute > max_min - 15 else "bottom right")
                                               for k, r in enumerate(g.itertuples())],
                                 textfont=dict(color=GOLD, size=11),
                                 hovertext=[h for h, isg in zip(hover, s.is_goal) if isg], hoverinfo="text"), row=1, col=1)
    if shots.empty:
        fig.add_annotation(text="no Understat shots for this match", xref="x domain", yref="y domain", x=0.5, y=0.5,
                           showarrow=False, font=dict(color=MUTED))
    e = ev[ev.action.ne("unknown") & ev.player_id.ne("")].copy()
    e["bin"] = (e.minute // BIN) * BIN
    ft = e[e.x >= 200 / 3].groupby(["bin", "team_id"]).size().unstack(fill_value=0)
    ft = ft.reindex(range(0, max_min + BIN, BIN), fill_value=0)
    h, a = ft.get(M["home"], 0), ft.get(M["away"], 0)
    tilt = ((h - a) / (h + a).replace(0, np.nan)).fillna(0)
    fig.add_trace(go.Bar(x=tilt.index + BIN / 2, y=tilt, width=BIN * 0.9, showlegend=False,
                         marker=dict(color=[M["colors"][M["home"]] if v >= 0 else M["colors"][M["away"]] for v in tilt]),
                         hovertext=[f"{int(b)}–{int(b) + BIN}': {M['names'][M['home']]} {int(hh)} · {M['names'][M['away']]} {int(aa)} final-third actions"
                                    for b, hh, aa in zip(tilt.index, np.asarray(h) if not np.isscalar(h) else [0]*len(tilt), np.asarray(a) if not np.isscalar(a) else [0]*len(tilt))],
                         hoverinfo="text"), row=2, col=1)
    for r in (1, 2):
        fig.add_vline(x=45, line=dict(color=LINE, width=1, dash="dash"), row=r, col=1)
    fig.update_yaxes(title_text="cumulative xG", gridcolor="#2a3140", rangemode="tozero", row=1, col=1)
    fig.update_yaxes(title_text="territory", range=[-1, 1], gridcolor="#2a3140", zerolinecolor=LINE, row=2, col=1)
    fig.update_xaxes(title_text="minute", range=[0, max_min], gridcolor="#2a3140", row=2, col=1)
    fig.update_xaxes(gridcolor="#2a3140", row=1, col=1)
    fig.add_annotation(text=f"↑ {M['names'][M['home']]}", xref="x2 domain", yref="y2 domain", x=0.01, y=0.95, showarrow=False, font=dict(size=10, color=M["colors"][M["home"]]), xanchor="left")
    fig.add_annotation(text=f"↓ {M['names'][M['away']]}", xref="x2 domain", yref="y2 domain", x=0.01, y=0.05, showarrow=False, font=dict(size=10, color=M["colors"][M["away"]]), xanchor="left")
    fig.update_layout(height=640, **BASE_LAYOUT, hovermode="closest")
    fig.update_layout(legend=dict(y=0.99, x=0.01, yanchor="top", orientation="v"))
    suptitle(fig, M["title"])
    note(fig, f"xG: gold.fct_shot (Understat) · territory: share of each team's actions in the attacking third per {BIN}-minute bin, gold.fct_event (WhoScored)")
    return fig


def plot_pass_map(M, player_id):
    ev = M["events"]
    p = M["players"].set_index("player_id").loc[player_id]
    passes = ev[(ev.player_id == player_id) & ev.action.isin(["pass", "cross"])].copy()
    metrics, pct = season_player_metrics(M["meta"].league, M["meta"].season)

    fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.17,
                        subplot_titles=[" ", " "])
    fig.update_layout(shapes=pitch_shapes("x", "y"))
    pitch_axes(fig, 1, 1)
    passes["ok"] = passes.success.fillna(False)
    for ok, color, label in ((True, OK, "completed"), (False, BAD, "incomplete")):
        d = passes[passes.ok == ok]
        hover = [f"{int(r.minute)}' · {r.action} · {label}" for r in d.itertuples()]
        add_arrows(fig, d, color, "x", "y", hover=hover, name=label, showlegend=True)
    n_ok = int(passes.ok.sum())
    mins = f" · {int(p.minutes_played)} min" if pd.notna(p.minutes_played) else ""
    fig.layout.annotations[0].text = (f"<b>{p.player_name} ({M['names'][p.team_id]}) · {len(passes)} passes, "
                                      f"{n_ok} completed ({n_ok / max(len(passes), 1):.0%}){mins}</b>")
    fig.add_annotation(text="attacking →", x=97, y=96, xref="x", yref="y", showarrow=False,
                       font=dict(size=10, color=MUTED), xanchor="right")
    if metrics is None or player_id not in metrics.index:
        fig.update_xaxes(visible=False, row=1, col=2)
        fig.update_yaxes(visible=False, row=1, col=2)
        fig.add_annotation(text="no season percentiles<br>(fct_player_match: <900 min or no data)",
                           xref="x2 domain", yref="y2 domain", x=0.5, y=0.5, showarrow=False, font=dict(color=MUTED))
    else:
        labels = list(metrics.columns)[::-1]
        pcs = [pct.loc[player_id, m] for m in labels]
        vals = [metrics.loc[player_id, m] for m in labels]
        fig.add_trace(go.Bar(y=labels, x=[100] * len(labels), orientation="h", marker=dict(color="#2a3140"),
                             hoverinfo="skip", showlegend=False, width=0.35), row=1, col=2)
        fig.add_trace(go.Bar(y=labels, x=pcs, orientation="h", width=0.35, showlegend=False,
                             marker=dict(color=[OK if v >= 50 else BAD for v in pcs]),
                             hovertext=[f"{m}: {v:.2f} → {pc:.0f}th percentile" for m, v, pc in zip(labels, vals, pcs)],
                             hoverinfo="text"), row=1, col=2)
        fig.add_trace(go.Scatter(x=[104] * len(labels), y=labels, mode="text", showlegend=False, hoverinfo="skip",
                                 text=[f"{v:.2f} · {pc:.0f}th" for v, pc in zip(vals, pcs)], textposition="middle right",
                                 textfont=dict(color=MUTED, size=11)), row=1, col=2)
        fig.update_layout(barmode="overlay")
        fig.update_xaxes(range=[0, 140], tickvals=[0, 25, 50, 75, 100], gridcolor="#2a3140", row=1, col=2)
        fig.update_yaxes(tickfont=dict(size=11), row=1, col=2)
        fig.add_vline(x=50, line=dict(color=LINE, width=1, dash="dot"), row=1, col=2)
        fig.layout.annotations[1].text = (f"<b>{M['meta'].league.split('-', 1)[1]} {M['meta'].season} percentiles · "
                                          f"{len(metrics)} players ≥ 900 min</b>")
    for ann, x in zip(fig.layout.annotations[:2], (0, fig.layout.xaxis2.domain[0])):
        ann.update(x=x, xanchor="left", font=dict(size=13, color=TEXT))
    fig.update_layout(height=560, **BASE_LAYOUT)
    note(fig, "Pass map: gold.fct_event (WhoScored) · percentiles: gold.fct_player_match, per 90 minutes")
    return fig


def plot_shots_entries(M, DEEP_M=20):
    ev, shots, names = M["events"], M["shots"], _names(M)
    entries = ev[ev.action.isin(["pass", "cross"]) & ev.end_x.notna()].copy()
    entries = entries[(dist_to_goal_m(entries.end_x, entries.end_y) <= DEEP_M)
                      & (dist_to_goal_m(entries.x, entries.y) > DEEP_M)]
    fig = pitch_subplots(["", ""], xlim=(45, 103))
    for i, team in enumerate((M["home"], M["away"])):
        col = i + 1
        xa, ya = axis_names(fig, 1, col)
        fig.add_shape(type="path", path=_ellipse_path(100, 50, DEEP_M * KX, DEEP_M * KY, np.pi / 2, 3 * np.pi / 2, 30),
                      line=dict(color=GOLD, width=1, dash="dash"), opacity=0.6, xref=xa, yref=ya)
        e = entries[entries.team_id == team].copy()
        e["ok"] = e.success.fillna(False)
        for ok, color, label in ((True, OK, "entry completed"), (False, BAD, "entry incomplete")):
            d = e[e.ok == ok]
            hover = [f"{names.get(r.player_id, r.player_id)} · {int(r.minute)}' · {r.action} · {label}" for r in d.itertuples()]
            add_arrows(fig, d, color, xa, ya, width=1.0, opacity=0.75, hover=hover, name=label, showlegend=(i == 0))
        s = shots[shots.team_id == team]
        hover = [f"{r.player_name} · {int(r.minute)}' · xG {r.xg:.2f} · {r.result} ({r.situation})" for r in s.itertuples()]
        fig.add_trace(go.Scatter(x=s.x, y=s.y, mode="markers", name="shot (size = xG)",
                                 marker=dict(size=8 + s.xg * 40, color=M["colors"][team], opacity=0.55, line=dict(color=BG, width=0.6)),
                                 hovertext=hover, hoverinfo="text", showlegend=(i == 0), legendgroup="shot"), row=1, col=col)
        g = s[s.is_goal]
        fig.add_trace(go.Scatter(x=g.x, y=g.y, mode="markers", name="goal",
                                 marker=dict(size=12 + g.xg * 40, symbol="star", color=GOLD, line=dict(color=BG, width=0.6)),
                                 hovertext=[h for h, isg in zip(hover, s.is_goal) if isg], hoverinfo="text",
                                 showlegend=(i == 0), legendgroup="goal"), row=1, col=col)
        n_ok = int(e.ok.sum())
        fig.layout.annotations[i].text = (f"<b>{M['names'][team]}: {len(s)} shots, {s.xg.sum():.2f} xG · "
                                          f"{len(e)} deep entries ({n_ok} completed)</b>")
    suptitle(fig, M["title"])
    note(fig, f"Shots: gold.fct_shot (Understat) · deep entries: passes/crosses ending within {DEEP_M} m of goal centre, gold.fct_event (WhoScored)")
    return fig


def plot_network(M, min_actions=25, min_edge=3):
    ev, players = M["events"], M["players"]
    fig = pitch_subplots(["", ""])
    for i, team in enumerate((M["home"], M["away"])):
        col = i + 1
        ev_t = ev[(ev.team_id == team) & ev.player_id.ne("")]
        pl = players[players.team_id == team]
        core = set(pl[pl.minutes_played >= 45].player_id) if pl.minutes_played.notna().any() \
            else set(pl[pl.actions >= min_actions].player_id)
        pos = ev_t[ev_t.player_id.isin(core)].groupby("player_id")[["x", "y"]].mean()
        names = pl.set_index("player_id").player_name
        nxt = ev.shift(-1)
        ok = ev_t[ev_t.action.eq("pass") & ev_t.success.fillna(False)].index
        edges = pd.DataFrame({"src": ev.loc[ok, "player_id"], "dst": nxt.loc[ok, "player_id"],
                              "same": nxt.loc[ok, "team_id"] == team})
        edges = edges[edges.same & edges.dst.ne("") & edges.src.isin(core) & edges.dst.isin(core)]
        edges = edges[edges.src != edges.dst].groupby(["src", "dst"]).size()
        edges = edges.groupby(lambda k: tuple(sorted(k))).sum() if len(edges) else edges
        n_pass = ev_t[ev_t.action.eq("pass")].groupby("player_id").size().reindex(pos.index, fill_value=0)
        color = M["colors"][team]
        mx, my, mh = [], [], []
        for (a, b), n in edges[edges >= min_edge].items():
            fig.add_trace(go.Scatter(x=[pos.x[a], pos.x[b]], y=[pos.y[a], pos.y[b]], mode="lines",
                                     line=dict(color=color, width=0.5 + n * 0.35), opacity=0.35,
                                     hoverinfo="skip", showlegend=False), row=1, col=col)
            mx.append((pos.x[a] + pos.x[b]) / 2)
            my.append((pos.y[a] + pos.y[b]) / 2)
            mh.append(f"{names.get(a, a)} ↔ {names.get(b, b)}: {int(n)} passes")
        fig.add_trace(go.Scatter(x=mx, y=my, mode="markers", marker=dict(size=10, opacity=0.001, color=color),
                                 hovertext=mh, hoverinfo="text", showlegend=False), row=1, col=col)
        fig.add_trace(go.Scatter(x=pos.x, y=pos.y, mode="markers+text", showlegend=False,
                                 marker=dict(size=12 + n_pass.values * 0.35, color=color, line=dict(color=BG, width=1)),
                                 text=[str(names.get(pid, pid)).split()[-1] for pid in pos.index], textposition="top center",
                                 textfont=dict(size=10, color=TEXT),
                                 hovertext=[f"{names.get(pid, pid)}: {int(n_pass[pid])} passes" for pid in pos.index],
                                 hoverinfo="text"), row=1, col=col)
        fig.layout.annotations[i].text = f"<b>{M['names'][team]} · pass network ({int(n_pass.sum())} passes)</b>"
    suptitle(fig, M["title"])
    note(fig, "gold.fct_event (WhoScored): node = mean position of a player's actions, edge = completed passes between the pair (≥ 3)")
    return fig


def plot_form(league, season, LAST_N=7):
    tm = season_team_matches(league, season).copy()
    fig = go.Figure()
    fig.update_layout(height=700, **BASE_LAYOUT)
    if tm.empty or tm.op_xg_for.sum() == 0:
        fig.add_annotation(text="no team/shot data for this season", xref="paper", yref="paper", x=0.5, y=0.5,
                           showarrow=False, font=dict(color=MUTED))
        return fig
    tm["is_recent"] = tm.groupby("team_id").cumcount(ascending=False) < LAST_N
    form = (tm.groupby(["team_id", "short_name", "is_recent"])[["op_xg_for", "op_xg_against"]]
            .mean().unstack("is_recent")).dropna()
    anns, hx, hy, ht, hc = [], [], [], [], []
    for (team, name), row in form.iterrows():
        x0, y0 = row[("op_xg_for", False)], row[("op_xg_against", False)]
        x1, y1 = row[("op_xg_for", True)], row[("op_xg_against", True)]
        c = OK if (x1 - y1) > (x0 - y0) else BAD
        anns.append(dict(x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y", showarrow=True,
                         arrowhead=2, arrowsize=1.2, arrowwidth=1.6, arrowcolor=c, text=""))
        anns.append(dict(x=x1, y=y1, text=f"<b>{name}</b>", showarrow=False, xanchor="left", xshift=8, yshift=6,
                         font=dict(size=11, color=TEXT)))
        hx.append(x1)
        hy.append(y1)
        hc.append(c)
        ht.append(f"<b>{name}</b><br>before: {x0:.2f} xG for · {y0:.2f} against<br>last {LAST_N}: {x1:.2f} xG for · {y1:.2f} against")
    fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", marker=dict(size=9, color=hc), hovertext=ht,
                             hoverinfo="text", showlegend=False))
    fig.update_layout(annotations=anns)
    fig.update_xaxes(title_text="open-play xG per game (for)", gridcolor="#2a3140")
    fig.update_yaxes(title_text="open-play xG per game (against) — inverted", autorange="reversed", gridcolor="#2a3140")
    suptitle(fig, f"{league.split('-', 1)[1]} {season} · form change: last {LAST_N} games (head) vs. games before (tail)")
    fig.add_annotation(text="green: xG difference improved · red: worsened", xref="paper", yref="paper", x=1, y=0,
                       showarrow=False, font=dict(size=10, color=MUTED), xanchor="right")
    note(fig, "gold.fct_team_match + gold.fct_shot (Understat, situation = open_play)")
    return fig

## Панель выбора

Лига → сезон → матч → команда → игрок. Смена матча перерисовывает все вкладки,
смена игрока — только «Pass map», смена сезона — ещё и «League form».

In [5]:
DEFAULT = {"league": "ITA-Serie A", "season": "2526", "match": "83708428"}

league_dd = w.Dropdown(options=LEAGUES, value=DEFAULT["league"] if DEFAULT["league"] in LEAGUES else LEAGUES[0],
                       description="Лига", layout=w.Layout(width="230px"))
season_dd = w.Dropdown(description="Сезон", layout=w.Layout(width="150px"))
match_dd = w.Dropdown(description="Матч", layout=w.Layout(width="520px"))
team_dd = w.Dropdown(description="Команда", layout=w.Layout(width="230px"))
player_dd = w.Dropdown(description="Игрок", layout=w.Layout(width="300px"))
status = w.HTML()

TABS = ["Losses", "xG timeline", "Pass map", "Shots & entries", "Pass network", "League form"]
outs = {t: w.Output() for t in TABS}
tabs = w.Tab(children=[outs[t] for t in TABS])
for i, t in enumerate(TABS):
    tabs.set_title(i, t)

_state = {"updating": False}


def show(name, make):
    out = outs[name]
    with out:
        out.clear_output(wait=True)
        try:
            display(make())
        except Exception as exc:  # чтобы одна вкладка не ломала остальные
            print(f"{name}: {type(exc).__name__}: {exc}")


def render_match():
    M = load_match(match_dd.value)
    status.value = f"<b>{M['title']}</b> · events: {len(M['events'])}, shots: {len(M['shots'])}"
    show("Losses", lambda: plot_losses(M))
    show("xG timeline", lambda: plot_timeline(M))
    show("Shots & entries", lambda: plot_shots_entries(M))
    show("Pass network", lambda: plot_network(M))
    render_player()


def render_player():
    if player_dd.value:
        show("Pass map", lambda: plot_pass_map(load_match(match_dd.value), player_dd.value))


def render_form():
    show("League form", lambda: plot_form(league_dd.value, season_dd.value))


def set_options(dd, options, value=None):
    dd.options = options
    values = [v for _, v in options]
    if values:
        dd.value = value if value in values else values[0]


def on_league(_=None):
    _state["updating"] = True
    seasons = seasons_of(league_dd.value)
    set_options(season_dd, [(s, s) for s in seasons], DEFAULT["season"])
    _state["updating"] = False
    on_season()


def on_season(_=None):
    _state["updating"] = True
    df = matches_of(league_dd.value, season_dd.value)
    set_options(match_dd, list(zip(df.label, df.match_id)), DEFAULT["match"])
    _state["updating"] = False
    render_form()
    on_match()


def on_match(_=None):
    if _state["updating"]:
        return
    _state["updating"] = True
    M = load_match(match_dd.value)
    set_options(team_dd, [(M["names"][t], t) for t in (M["home"], M["away"])])
    fill_players(M)
    _state["updating"] = False
    render_match()


def fill_players(M):
    pl = M["players"][M["players"].team_id == team_dd.value].sort_values("passes", ascending=False)
    set_options(player_dd, [(f"{r.player_name} ({int(r.passes)} passes)", r.player_id) for r in pl.itertuples()])


def on_team(_=None):
    if _state["updating"]:
        return
    _state["updating"] = True
    fill_players(load_match(match_dd.value))
    _state["updating"] = False
    render_player()


def on_player(_=None):
    if not _state["updating"]:
        render_player()


league_dd.observe(on_league, names="value")
season_dd.observe(on_season, names="value")
match_dd.observe(on_match, names="value")
team_dd.observe(on_team, names="value")
player_dd.observe(on_player, names="value")

display(w.VBox([w.HBox([league_dd, season_dd, match_dd]), w.HBox([team_dd, player_dd]), status, tabs]))
on_league()

## Что ещё можно построить на этих же данных

- Карта прессинга и возвратов мяча (`ball_recovery`, `tackle`, `interception`) по зонам.
- Сравнение двух игроков «радаром» по перцентилям из вкладки Pass map.
- Хронология сезона: xG за/против по турам со скользящим средним (`fct_team_match`).
- Тепловая карта касаний игрока и «сонары» передач по направлениям.
- Модель угрозы (xT) поверх `fct_event` — тогда моментум станет «не-ударным xG».